In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
import os
from torch import Tensor
from torch.utils.data import DataLoader
import faiss
import json
from beir.datasets.data_loader import GenericDataLoader

In [2]:
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
corpus, _, _ = GenericDataLoader(data_folder=data_dir).load(split="train")
passages = [corpus[doc_id]["text"] for doc_id in corpus][:10]

  0%|          | 0/8841823 [00:00<?, ?it/s]

In [3]:
# Selecting Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loading Passage Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
passage_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/RAG/model_weights/passage_encoder"
).to(device)
passage_encoder.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [4]:
# Custom MSMARCO class
class MSMARCO:
    def __init__(self, passages):
    
        '''
        DataLoader for Vector Database
        '''

        self.passages = passages
    
    def __len__(self):
        return len(self.passages)
    
    def __getitem__(self, idx):
        passage = self.passages[idx]
        return {"passage": passage}

In [5]:
dataset = MSMARCO(passages)

In [7]:
dataloader = DataLoader(dataset,
                        batch_size=32,
                        shuffle=False,
                        num_workers=4,
                        pin_memory=True)

In [8]:
dataloader = DataLoader(dataset, batch_size=64)

In [11]:
for batch in dataloader:
    print(batch['passage'])
    print(len(batch['passage']))
    break


['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.', 'The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.', 'Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this project would forever change the world forever making it known that something this powerful can be manmade.', 'The Manhattan Project was the name for a project conducted during World War II, to develop the first atomic bomb. It refers specifically to the period of the project from 194 â\x80¦ 2-1946 under the control of the U.S. Army Corps of Eng

In [20]:
index = faiss.IndexFlatIP(768)
meta_file = open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl", "w")
global_idx = 0

for batch in dataloader:
    with torch.no_grad():
        inputs = tokenizer(
            batch['passage'],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(device)

        emb = passage_encoder(**inputs).last_hidden_state[:, 0]

    emb = emb.float().cpu().numpy()
    emb = np.ascontiguousarray(emb, dtype=np.float32)
    faiss.normalize_L2(emb)
    index.add(emb)

    for i, passage in enumerate(batch['passage']):
        meta = {
            'passage': passage,
            'idx': global_idx
        }
        meta_file.write(json.dumps(meta) + "\n")
        global_idx += 1

    del emb, inputs
    torch.cuda.empty_cache()

faiss.write_index(index, "/work/mbouthil/projects/research_project/RAG/retrieval_data/passage.index")

In [24]:
print(index.ntotal)
print(sum(1 for _ in open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl")))

10
10
